# HomeSolar - Exploratory Data Analysis

This notebook explores `data/processed/household_energy_dataset.csv` before model training.

It covers electricity-reading distributions, hourly and weekday patterns, weather relationships, missing data and unusual readings. Because the processed file contains about 3.9 million rows, plots use a reproducible chunk sample to keep memory use manageable; row counts and basic data-quality totals are scanned across the full CSV.

**Interpretation notes**
- `t_kWh` is treated as energy consumed in one 3-minute interval, consistent with the supplied diagnostic sample where it varies up/down and is close to an approximate V-I-times-interval calculation. This remains a dataset interpretation, not a manufacturer calibration certificate.
- The `BRxx` to `Mxx` weather mapping used in the current dataset was assigned sequentially by project decision. It is not a verified household GPS mapping.
- Weather data is matched at hourly resolution and reused for multiple 3-minute electricity readings.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

# Resolve project root whether Jupyter starts at Home-Solar/ or notebooks/.
def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed" / "household_energy_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/processed/household_energy_dataset.csv. Start Jupyter from Home-Solar/.")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data/processed/household_energy_dataset.csv"
FIGURE_DIR = PROJECT_ROOT / "reports/eda"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)


In [ ]:
# Scan the full CSV and take a reproducible sample for plots.
# Each chunk contributes up to SAMPLE_PER_CHUNK rows; the final sample is capped.
SAMPLE_PER_CHUNK = 10_000
MAX_SAMPLE_ROWS = 200_000
CHUNK_SIZE = 150_000
USECOLS = [
    "meter", "timestamp", "timestamp_utc", "t_kWh", "avg_voltage_v", "avg_current_a",
    "temperature_2m", "relative_humidity_2m", "cloud_cover", "dew_point_2m", "shortwave_radiation",
    "direct_radiation", "global_tilted_irradiance", "weather_match", "weather_available",
    "hour", "day_of_week", "month", "split"
]

header = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
usecols = [c for c in USECOLS if c in header]
rng = np.random.default_rng(42)
samples = []
full_rows = 0
full_missing_t_kwh = 0
full_missing_weather = 0
for chunk in pd.read_csv(DATA_PATH, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False):
    full_rows += len(chunk)
    full_missing_t_kwh += int(chunk["t_kWh"].isna().sum())
    weather_col = "weather_match" if "weather_match" in chunk.columns else "weather_available"
    if weather_col in chunk.columns:
        if weather_col == "weather_match":
            full_missing_weather += int((~chunk[weather_col].astype("string").str.lower().isin(["true", "1", "yes"])).sum())
        else:
            full_missing_weather += int(pd.to_numeric(chunk[weather_col], errors="coerce").fillna(0).eq(0).sum())
    if len(chunk):
        n = min(SAMPLE_PER_CHUNK, len(chunk))
        samples.append(chunk.sample(n=n, random_state=int(rng.integers(0, 2**31 - 1))))

df = pd.concat(samples, ignore_index=True)
if len(df) > MAX_SAMPLE_ROWS:
    df = df.sample(n=MAX_SAMPLE_ROWS, random_state=42).reset_index(drop=True)

for col in ["t_kWh", "avg_voltage_v", "avg_current_a", "temperature_2m", "relative_humidity_2m", "cloud_cover",
            "dew_point_2m", "shortwave_radiation", "direct_radiation", "global_tilted_irradiance"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

time_col = "timestamp_local" if "timestamp_local" in df.columns else ("timestamp" if "timestamp" in df.columns else "timestamp_utc")
df["analysis_timestamp"] = pd.to_datetime(df[time_col], errors="coerce", utc=True, format="mixed")
if "hour" not in df.columns:
    df["hour"] = df["analysis_timestamp"].dt.tz_convert("Asia/Kolkata").dt.hour
if "day_of_week" not in df.columns:
    df["day_of_week"] = df["analysis_timestamp"].dt.tz_convert("Asia/Kolkata").dt.day_of_week

print(f"Full dataset rows: {full_rows:,}")
print(f"Rows sampled for visual analysis: {len(df):,}")
print("Unique households in sample:", df["meter"].nunique())
print("Missing t_kWh across full CSV:", f"{full_missing_t_kwh:,}")
print(df.head())


In [ ]:
# Distribution of interval electricity energy (kWh per 3-minute reading).
energy = df["t_kWh"].dropna()
q995 = energy.quantile(0.995)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(energy.clip(upper=q995), bins=60)
ax.set_title("Distribution of 3-minute household electricity readings")
ax.set_xlabel("Energy per interval (kWh; values clipped at sample 99.5th percentile for display)")
ax.set_ylabel("Sample row count")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_energy_distribution.png", dpi=160)
plt.show()

print(energy.describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.995]).to_frame("t_kWh"))


In [ ]:
# Average interval energy by hour of day.
hour_profile = df.groupby("hour", dropna=True)["t_kWh"].mean().reindex(range(24))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(hour_profile.index, hour_profile.values, marker="o")
ax.set_xticks(range(24))
ax.set_xlabel("Hour of day (India local time)")
ax.set_ylabel("Mean energy per 3-minute interval (kWh)")
ax.set_title("Average household electricity use by hour")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_hourly_profile.png", dpi=160)
plt.show()



In [ ]:
# Average interval energy by weekday (0=Monday, 6=Sunday).
weekday_profile = df.groupby("day_of_week", dropna=True)["t_kWh"].mean().reindex(range(7))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(weekday_profile.index, weekday_profile.values)
ax.set_xticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
ax.set_ylabel("Mean energy per 3-minute interval (kWh)")
ax.set_title("Average household electricity use by day of week")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_weekday_profile.png", dpi=160)
plt.show()


In [ ]:
# Inspect relationships between weather and interval electricity use.
weather_pairs = [
    ("temperature_2m", "Temperature at 2 m (°C)"),
    ("relative_humidity_2m", "Relative humidity (%)"),
    ("cloud_cover", "Cloud cover (%)"),
    ("shortwave_radiation", "Shortwave radiation (W/m², as provided)")
]
weather_pairs = [(c, label) for c, label in weather_pairs if c in df.columns]
fig, axes = plt.subplots(1, max(1, len(weather_pairs)), figsize=(4.5 * max(1, len(weather_pairs)), 4.5))
for ax, (column, label) in zip(axes, weather_pairs):
    valid = df[[column, "t_kWh"]].dropna()
    if len(valid) > 0:
        hb = ax.hexbin(valid[column], valid["t_kWh"], gridsize=35, mincnt=1)
        ax.set_xlabel(label)
        ax.set_ylabel("Energy per 3-minute interval (kWh)")
        fig.colorbar(hb, ax=ax, label="Sample count")
    else:
        ax.set_title(f"No valid values: {column}")
fig.suptitle("Weather and household electricity use (sampled observations)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_weather_relationships.png", dpi=160)
plt.show()


In [ ]:
# Correlation matrix for numeric electricity, weather and time features.
correlation_cols = [
    "t_kWh", "avg_voltage_v", "avg_current_a", "frequency_hz", "temperature_2m",
    "relative_humidity_2m", "cloud_cover", "dew_point_2m", "shortwave_radiation",
    "direct_radiation", "global_tilted_irradiance", "hour", "day_of_week"
]
correlation_cols = [c for c in correlation_cols if c in df.columns]
corr = df[correlation_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=75, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
ax.set_title("Correlation matrix (sampled rows)")
fig.colorbar(image, ax=ax, label="Pearson correlation")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_correlation_matrix.png", dpi=160)
plt.show()


In [ ]:
# Flag unusual readings using the IQR rule. This is an inspection aid, not an automatic deletion rule.
q1 = energy.quantile(0.25)
q3 = energy.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outlier_mask = (df["t_kWh"] < lower_bound) | (df["t_kWh"] > upper_bound)
print("IQR lower bound:", lower_bound)
print("IQR upper bound:", upper_bound)
print("Sample outlier rows:", int(outlier_mask.sum()))
print("Sample outlier share (%):", round(float(outlier_mask.mean() * 100), 3))
if "meter" in df.columns:
    print("Outlier counts by household (sample):")
    print(df.loc[outlier_mask].groupby("meter").size().sort_values(ascending=False).rename("outlier_rows"))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(energy.dropna(), vert=False, showfliers=False)
ax.set_xlabel("Energy per 3-minute interval (kWh)")
ax.set_title("Electricity reading distribution (boxplot; fliers hidden for readability)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "06_energy_boxplot.png", dpi=160)
plt.show()


## EDA conclusions to record

After running the notebook, write down the patterns that are actually visible in your plots. In particular:

1. Whether consumption has morning/evening peaks.
2. Whether weekday and weekend profiles differ.
3. Whether temperature or humidity has a visible relationship with consumption.
4. Whether solar radiation aligns with daytime usage patterns.
5. Whether high-IQR readings are genuine peaks or meter/data-quality issues.

Do not automatically remove high readings based only on IQR. Confirm them against voltage/current and the source data first. The notebook saves charts under `reports/eda/`.
